In [ ]:
!pip install --user -e /scratch/v46/sg3241/tmp/PyFLEXTRKR/
!pip install --user colormaps cmweather

In [ ]:
# # SPECIAL METHOD TO IMPORT LEROI RADAR GRIDDING PACKAGE FROM LOCAL DIRECTORY
# import sys
# sys.path.append('/home/563/sg3241/PyFLEXTRKR/pyflextrkr')

import pyflextrkr
print(pyflextrkr.__file__)

In [ ]:
from pathlib import Path
import os

# Specify directory for the demo data
dir_demo = '/scratch/v46/sg3241/tmp/PyFLEXTRKR/'

# Example config file name
config_demo = 'config_mcs_demo.yml'

# Demo input data directory
dir_input = dir_demo + 'input/'

# Create the demo directory
Path(dir_input).mkdir(parents=True, exist_ok=True)

# PyFLEXTRKR repo location — adjust this to wherever you cloned it
pyflextrkr_repo = '/scratch/v46/sg3241/tmp/PyFLEXTRKR/'

print(f"Demo dir:        {dir_demo}")
print(f"Input dir:       {dir_input}")
print(f"PyFLEXTRKR repo: {pyflextrkr_repo}")

In [ ]:
!cd /scratch/v46/sg3241/tmp/ && git clone https://github.com/FlexTRKR/PyFLEXTRKR.git

In [ ]:
print('Downloading demo input data ...')
!wget https://portal.nersc.gov/project/m1867/PyFLEXTRKR/sample_data/tb_pcp/gpm_tb_imerg.tar.gz -O {dir_input}gpm_tb_imerg.tar.gz

In [ ]:
print('Extracting demo input data ...')
!tar -xvzf {dir_input}gpm_tb_imerg.tar.gz -C {dir_input}

# Remove downloaded tar file
!rm -fv {dir_input}gpm_tb_imerg.tar.gz

# Sanity check: list what got extracted
!ls -la {dir_input}

In [ ]:
# Move into the config directory of the PyFLEXTRKR repo
os.chdir(pyflextrkr_repo + 'config/')
print(f"Current dir: {os.getcwd()}")

# Sanity check that the example file exists
!ls config_imerg_mcs_tbpf_example.yml

In [ ]:
# Build the sed command — easier to do this in Python than try to escape paths in shell
dir_input_escaped = dir_input.replace('/', r'\/')
dir_demo_escaped = dir_demo.replace('/', r'\/')

sed_cmd = (
    f"sed 's/INPUT_DIR/{dir_input_escaped}/g;s/TRACK_DIR/{dir_demo_escaped}/g' "
    f"config_imerg_mcs_tbpf_example.yml > {config_demo}"
)
print(sed_cmd)
!{sed_cmd}

print(f'Created new config file: {config_demo}')
# Inspect it to make sure substitution worked
!head -30 {config_demo}

In [ ]:
%%time
print('Running PyFLEXTRKR ...')
!python {pyflextrkr_repo}runscripts/run_mcs_tbpf.py {config_demo}
print('Tracking is done.')


feature stats = feature tracking (data set)

In [ ]:
quicklook_dir = dir_demo + 'quicklooks_trackpaths/'

print('Making quicklook plots ...')
!python {pyflextrkr_repo}Analysis/plot_subset_tbpf_mcs_tracks_demo.py \
    -s '2019-01-25T00' -e '2019-01-27T00' \
    -c {config_demo} \
    -o vertical -p 1 --figsize 10 8 \
    --output {quicklook_dir}

print(f'View quicklook plots here: {quicklook_dir}')
!ls {quicklook_dir} | head -20

In [ ]:
print('Making animation from quicklook plots ...')
!ffmpeg -framerate 2 -pattern_type glob -i '{quicklook_dir}*.png' \
    -c:v libx264 -r 10 -crf 20 -pix_fmt yuv420p \
    -y {quicklook_dir}quicklook_animation.mp4

print(f'View animation here: {quicklook_dir}quicklook_animation.mp4')

In [ ]:
# PROBABLY WHAT YOU HAVE TO DOWNLOAD AS EXAMPLE RADAR DATA TO PLAY WITH 

# Download sample WRF Tb+Precipitation data:
echo 'Downloading demo input data ...'
wget https://portal.nersc.gov/project/m1867/PyFLEXTRKR/sample_data/tb_radar/wrf_tbradar.tar.gz \
  -O ${dir_input}/wrf_tbradar.tar.gz

In [ ]:
%%bash
dir_input=/scratch/v46/sg3241/tmp/

mkdir -p ${dir_input}

echo 'Downloading demo input data ...'
wget -c --show-progress \
  https://portal.nersc.gov/project/m1867/PyFLEXTRKR/sample_data/tb_radar/wrf_tbradar.tar.gz \
  -O ${dir_input}/wrf_tbradar.tar.gz

tar -xzf ${dir_input}/wrf_tbradar.tar.gz -C ${dir_input}


In [ ]:
%%time
print('Running PyFLEXTRKR ...')
!python {pyflextrkr_repo}runscripts/run_mcs_tbpfradar3d_wrf.py {config_demo}
print('Tracking is done.')